In [0]:
%run /Configurations/Init_Scripts

-1
-1
0


jdbc:sqlserver://usazrsqld1027.database.windows.net:1433;database=CS-DataMonitoring;queryTimeout=30


In [0]:
dbutils.widgets.text("job_id","-1")
JobId = dbutils.widgets.get("job_id")

dbutils.widgets.text("run_id","-1")
PipelineRunId = dbutils.widgets.get("run_id")

ConfigID=dbutils.widgets.text("ConfigID","44")
ConfigID=dbutils.widgets.get("ConfigID")

dbutils.widgets.text('sourceTypeId','2')
sourceTypeId = dbutils.widgets.get('sourceTypeId')

dbutils.widgets.text('CreatedBy','ADB_Ingestion_SoldTo')  
CreatedBy = dbutils.widgets.get('CreatedBy')

dbutils.widgets.text("CatalogName", "cd_prod")
CatalogName = dbutils.widgets.get("CatalogName")

dbutils.widgets.text('ExternalLocation_raw',"/mntprod_raw")
ExternalLocation_raw = dbutils.widgets.get('ExternalLocation_raw')

dbutils.widgets.text("ExternalLocationName_silver", "/mntprod_silver")
ExternalLocationName_silver = dbutils.widgets.get("ExternalLocationName_silver")
target_table_path = ExternalLocation_silver+'/dimcustomerhierarchy'

UpdatedBy = 'ADB_Ingestion_SoldTo'  

In [0]:
database_host_url = dbutils.secrets.get(scope="ABV_AKV_ADB_SCOPE", key="SnowflakeURL")
username = dbutils.secrets.get(scope="ABV_AKV_ADB_SCOPE", key="SnowflakeUserName")
password = dbutils.secrets.get(scope="ABV_AKV_ADB_SCOPE", key="SnowflakePassword")
database_name = "AES_DW"
schema_name = "dw"
warehouse_name = "AES_ELT_WH_DEV"
table_name = "DIM_CUSTOMER_HIERARCHY"

In [0]:
# target_table_path = '/mnt/silver/dimcustomerhierarchy'
target_table = CatalogName+'silverzone.dimcustomerhierarchy'
checkpoint_column_from_target = 'SrcLastUpdatedDate'
default_date = '1900-01-01 00:00:00'

In [0]:
from pyspark.sql.utils import AnalysisException
from delta.tables import DeltaTable

try: 
    last_checkpoint = spark.sql(f"""
                              SELECT max({checkpoint_column_from_target}) as last_updated FROM {target_table} 
                              """).first()['last_updated']
except AnalysisException:
    last_checkpoint = default_date
print(last_checkpoint)    

1900-01-01 00:00:00


In [0]:
# Read Data from the source
df_source = (spark.read
  .format("snowflake")
  # .option("dbTable", table_name)
  .option("sfUrl", database_host_url)
  .option("sfUser", username)
  .option("sfPassword", password)
  .option("sfDatabase", database_name)
  .option("sfSchema", schema_name)
  .option("sfWarehouse", warehouse_name)
  .option("query", f"""SELECT * FROM {table_name} WHERE LAST_UPDATED_DATE > TO_TIMESTAMP('{last_checkpoint}') AND SOURCE_SYSTEM_NAME = 'SAP' """)
  .load()
)

In [0]:
window_spec = Window.partitionBy("ShipToAccountID", "SoldToAccountID", "PayerAccountID", "EffectiveDate").orderBy(col("TerminationDate").desc())

incremental_df = (df_source
                  .selectExpr("CUSTOMER_SHIPTO_ID as ShipToAccountID", 
                              "CUSTOMER_SOLDTO_ID as SoldToAccountID", 
                              "CUSTOMER_PAYER_ID as PayerAccountID", 
                              "EFFECTIVE_BEGIN_DATE as EffectiveDate",
                              "EFFECTIVE_END_DATE as TerminationDate",
                              "CURRENT_FLAG as CurrentFlag",
                              "LAST_UPDATED_DATE as SrcLastUpdatedDate"
                              )
                  .withColumn("row_num", row_number().over(window_spec))
                  .filter("row_num = 1")
                  .drop("row_num")
                  .withColumn('CreatedBy', lit(CreatedBy))
                  .withColumn('CreatedDate', current_timestamp())
                  .withColumn('UpdatedBy', lit(UpdatedBy))
                  .withColumn('UpdatedDate', current_timestamp())
                )

In [0]:
#Creating Log Entry
log_dict = {
"ConfigID" : ConfigID,
"SourceTypeID" : sourceTypeId,
"Source" : "AES_DW.DW.DIM_CUSTOMER_HIERARCHY",
"SourceFileName" : "",
"Destination" : "silverzone.dimcustomerhierarchy",
"DestinationFileName" : "",
"Run_ID": str(PipelineRunId),
"Job_ID": str(JobId)
}
audit_df = spark.createDataFrame([log_dict])

In [0]:

try:
    delta_target = DeltaTable.forName(spark, target_table)
    delta_target.alias("tgt").merge(
        incremental_df.alias("src"),
        """
        coalesce(tgt.ShipToAccountID, '') = coalesce(src.ShipToAccountID, '')
        AND coalesce(tgt.SoldToAccountID, '') = coalesce(src.SoldToAccountID, '')
        AND coalesce(tgt.PayerAccountID, '') = coalesce(src.PayerAccountID, '')
        AND tgt.EffectiveDate = src.EffectiveDate
        """
    ).whenMatchedUpdate(
        condition="""
        tgt.TerminationDate <> src.TerminationDate
        OR tgt.CurrentFlag <> src.CurrentFlag
        """,
        set={
            "TerminationDate": "src.TerminationDate",
            "CurrentFlag": "src.CurrentFlag",
            "SrcLastUpdatedDate": "src.SrcLastUpdatedDate",
            "UpdatedBy": "src.UpdatedBy",
            "UpdatedDate": "src.UpdatedDate"
        }
    ).whenNotMatchedInsert(
        values={
            "ShipToAccountID": "src.ShipToAccountID",
            "SoldToAccountID": "src.SoldToAccountID",
            "PayerAccountID": "src.PayerAccountID",
            "EffectiveDate": "src.EffectiveDate",
            "TerminationDate": "src.TerminationDate",
            "CurrentFlag": "src.CurrentFlag",
            "SrcLastUpdatedDate": "src.SrcLastUpdatedDate",
            "CreatedBy": "src.CreatedBy",
            "CreatedDate": "src.CreatedDate",
            "UpdatedBy": "src.UpdatedBy",
            "UpdatedDate": "src.UpdatedDate"
        }
    ).execute()
    logIntoPromotionLogTable(audit_df, CreatedBy, "Succeeded")
except Exception as e:
    logIntoPromotionLogTable(audit_df, CreatedBy, "Failed", str(e))
    raise e

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6037684408555264>, line 41
     39 except Exception as e:
     40     logIntoPromotionLogTable(audit_df, CreatedBy, "Failed", str(e))
---> 41     raise e

File <command-6037684408555264>, line 37
      1 try:
      2     delta_target = DeltaTable.forName(spark, target_table)
      3     delta_target.alias("tgt").merge(
      4         incremental_df.alias("src"),
      5         """
      6         coalesce(tgt.ShipToAccountID, '') = coalesce(src.ShipToAccountID, '')
      7         AND coalesce(tgt.SoldToAccountID, '') = coalesce(src.SoldToAccountID, '')
      8         AND coalesce(tgt.PayerAccountID, '') = coalesce(src.PayerAccountID, '')
      9         AND tgt.EffectiveDate = src.EffectiveDate
     10         """
     11     ).whenMatchedUpdate(
     12         condition="""
     13         tgt.TerminationDate <> src.